# Predicting F1 Tyre Degradation to Optimize Pit Stop Strategy
**Kevin Zong (115265318) & Michael Obajemu (114202291)**

This notebook demonstrates progress since the proposal:
- FastF1 pipeline running and data pulled for the 2024–2025 seasons
- Data cleaning with flagged removal of safety-car laps, in/out laps, and anomalies
- Feature engineering: fuel-corrected lap time, lap time delta, tyre-life ratio
- Exploratory visualizations: distributions, degradation curves by compound, correlation matrix

In [1]:
# !pip install fastf1

import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import gaussian_kde
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

fastf1.Cache.enable_cache('fastf1_cache')

print('Dependencies loaded successfully.')

Dependencies loaded successfully.


## 1. Data Collection

We pull race session lap data across the 2024 and 2025 F1 seasons using FastF1. Each row contains lap time, tyre compound, tyre life, stint number, pit events, and weather conditions.

**Season scope:**
- **2024** — 24 rounds, full season (training set)
- **2025** — 24 rounds, full season (validation set)

We use a representative 8-race subset spread across both seasons for development speed. The full pull (~50,000 clean laps) will be run before final modeling.

> **Note:** First run downloads and caches data locally — subsequent runs load from cache and are much faster.

In [ ]:
# 4 races from 2024, 4 from 2025 — spread across circuit types
ROUNDS = [
    # 2024 season
    (2024, 5,  'Miami'),
    (2024, 9,  'Canada'),
    (2024, 16, 'Italy'),
    (2024, 20, 'United States'),
    # 2025 season
    (2025, 1,  'Australia'),
    (2025, 2,  'China'),
    (2025, 3,  'Japan'),
    (2025, 4,  'Bahrain'),
]

all_laps = []

for year, rnd, name in ROUNDS:
    try:
        session = fastf1.get_session(year, rnd, 'R')
        session.load(telemetry=False, weather=True, messages=False)
        laps = session.laps.copy()
        laps['Race']  = name
        laps['Year']  = year
        laps['Round'] = rnd

        if session.weather_data is not None and not session.weather_data.empty:
            laps['AirTemp']   = session.weather_data['AirTemp'].mean()
            laps['TrackTemp'] = session.weather_data['TrackTemp'].mean()
        else:
            laps['AirTemp']   = np.nan
            laps['TrackTemp'] = np.nan

        all_laps.append(laps)
        print(f'  Loaded {name} {year}: {len(laps)} laps')
    except Exception as e:
        print(f'  Failed to load {name} {year}: {e}')

raw_df = pd.concat(all_laps, ignore_index=True)
print(f'\nRaw dataset: {len(raw_df):,} laps across {raw_df["Race"].nunique()} races')
print(f'Seasons:     {sorted(raw_df["Year"].unique())}')

## 2. Data Cleaning

We remove laps that don't reflect genuine tyre degradation:
- **In-laps / out-laps** — distorted by pit-lane speed limits
- **Safety car / VSC / red flag laps** — forced pace reductions, not tyre-driven
- **Wet/intermediate compounds** — out of scope for this model
- **Missing key fields** — unusable for modeling
- **Statistical outliers** — laps > 3 std deviations from per-race mean

Row counts are tracked at each step.

In [ ]:
df = raw_df.copy()
counts = {'Raw': len(df)}

# Step 1: Drop in-laps and out-laps
df = df[~df['PitInTime'].notna() & ~df['PitOutTime'].notna()]
counts['After removing in/out laps'] = len(df)

# Step 2: Remove safety car, VSC, red flag laps via TrackStatus
if 'TrackStatus' in df.columns:
    df = df[~df['TrackStatus'].astype(str).str.contains('4|5|6', na=False)]
counts['After removing SC/VSC/red flag laps'] = len(df)

# Step 3: Keep only dry compounds
df = df[df['Compound'].isin(['SOFT', 'MEDIUM', 'HARD'])]
counts['After keeping dry compounds only'] = len(df)

# Step 4: Drop rows missing key fields
df = df.dropna(subset=['LapTime', 'TyreLife', 'Compound', 'Driver'])
counts['After dropping missing values'] = len(df)

# Step 5: Convert LapTime to seconds, drop erroneous laps
df['LapTimeSec'] = df['LapTime'].dt.total_seconds()
df = df[df['LapTimeSec'] > 60]
counts['After removing sub-60s laps'] = len(df)

# Step 6: Remove per-race outliers (> 3 std from race mean)
race_stats = df.groupby('Race')['LapTimeSec'].agg(['mean', 'std']).reset_index()
df = df.merge(race_stats, on='Race')
df = df[np.abs(df['LapTimeSec'] - df['mean']) <= 3 * df['std']]
df = df.drop(columns=['mean', 'std'])
counts['After removing outliers (>3 std)'] = len(df)

# Summary
print('Cleaning summary:')
print('-' * 52)
prev = None
for step, n in counts.items():
    removed = f'  (-{prev - n:,} removed)' if prev is not None else ''
    print(f'  {step:<42} {n:>6,}{removed}')
    prev = n
print(f'\nFinal clean dataset: {len(df):,} laps')
print(f'By season:')
print(df.groupby('Year').size().to_string())

## 3. Feature Engineering

| Feature | Description |
|---|---|
| `LapTimeDelta` | Lap time minus per-driver per-stint median — captures degradation relative to each driver's fresh pace |
| `FuelCorrectedLap` | Crude fuel correction (~0.03s per lap lighter) |
| `TyreLifeRatio` | Tyre life normalized 0–1 within each stint |
| `CompoundEncoded` | Ordinal: SOFT=0, MEDIUM=1, HARD=2 |
| `StintLength` | Total laps in that driver's stint |

In [ ]:
# Lap time delta relative to each driver's per-stint median
stint_medians = df.groupby(['Race', 'Driver', 'Stint'])['LapTimeSec'].transform('median')
df['LapTimeDelta'] = df['LapTimeSec'] - stint_medians

# Fuel correction
df['FuelCorrectedLap'] = df['LapTimeSec'] - (0.03 * df['LapNumber'])

# Tyre life ratio within each stint
stint_max_life = df.groupby(['Race', 'Driver', 'Stint'])['TyreLife'].transform('max')
df['TyreLifeRatio'] = df['TyreLife'] / stint_max_life.replace(0, np.nan)

# Compound encoding
df['CompoundEncoded'] = df['Compound'].map({'SOFT': 0, 'MEDIUM': 1, 'HARD': 2})

# Stint length
df['StintLength'] = df.groupby(['Race', 'Driver', 'Stint'])['LapTimeSec'].transform('count')

print(f'Feature engineering complete. Dataset shape: {df.shape}')
df[['Race', 'Year', 'Driver', 'Compound', 'TyreLife', 'TyreLifeRatio',
    'LapTimeSec', 'LapTimeDelta', 'StintLength']].head(10)

## 4. Exploratory Visualizations

### 4.1 Distribution of Lap Times & Lap Time Delta by Compound

In [ ]:
compound_colors = {'SOFT': '#e63946', 'MEDIUM': '#f4a261', 'HARD': '#457b9d'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Lap time histogram, shaded by season
for year, color in [(2024, '#2196F3'), (2025, '#FF9800')]:
    subset = df[df['Year'] == year]['LapTimeSec']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, edgecolor='white',
                 linewidth=0.4, label=str(year))
axes[0].set_xlabel('Lap Time (seconds)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Lap Times by Season')
axes[0].legend(title='Season')

# KDE of lap time delta by compound
for compound, color in compound_colors.items():
    vals = df[df['Compound'] == compound]['LapTimeDelta'].dropna()
    vals = vals[np.isfinite(vals)]
    if len(vals) > 10:
        kde = gaussian_kde(vals, bw_method=0.3)
        x = np.linspace(vals.min(), vals.max(), 300)
        axes[1].plot(x, kde(x), label=compound, color=color, linewidth=2)
        axes[1].fill_between(x, kde(x), alpha=0.15, color=color)

axes[1].axvline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.6)
axes[1].set_xlabel('Lap Time Delta vs. Stint Median (s)')
axes[1].set_ylabel('Density')
axes[1].set_title('KDE: Lap Time Delta by Compound (2024–2025)')
axes[1].legend(title='Compound')

plt.tight_layout()
plt.savefig('plot_01_distributions.png', bbox_inches='tight')
plt.show()
print('\nInterpretation: Lap time distributions overlap between seasons (left), confirming'
      ' the data is comparable year-over-year. KDE shows SOFT tyres produce the widest'
      ' delta spread — higher upside early in a stint but sharper drop-off later.')

### 4.2 Tyre Life Distribution by Compound

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

compound_order = ['SOFT', 'MEDIUM', 'HARD']
colors = ['#e63946', '#f4a261', '#457b9d']
data_to_plot = [df[df['Compound'] == c]['TyreLife'].dropna() for c in compound_order]

bp = ax.boxplot(data_to_plot, patch_artist=True,
                medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticklabels(compound_order)
ax.set_xlabel('Compound')
ax.set_ylabel('Tyre Life (laps)')
ax.set_title('Tyre Life Distribution by Compound (2024–2025)')
plt.tight_layout()
plt.savefig('plot_02_tyre_life_dist.png', bbox_inches='tight')
plt.show()
print('\nInterpretation: HARD tyres are used for significantly longer stints than SOFT.'
      ' MEDIUM shows the widest spread, reflecting its use in both short and long strategies.')

### 4.3 Degradation Curves: Lap Time Delta vs. Tyre Life by Compound

The core visual of this project — does tyre degradation show a measurable, compound-specific signal?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, year in zip(axes, [2024, 2025]):
    year_df = df[df['Year'] == year]
    for compound, color in compound_colors.items():
        subset = year_df[year_df['Compound'] == compound]
        subset = subset[subset['TyreLife'] <= 40]
        binned = subset.groupby('TyreLife')['LapTimeDelta'].agg(['mean', 'sem']).reset_index()
        binned = binned[binned['TyreLife'] >= 2]
        if len(binned) >= 3:
            ax.plot(binned['TyreLife'], binned['mean'],
                    label=compound, color=color, linewidth=2.2)
            ax.fill_between(binned['TyreLife'],
                            binned['mean'] - binned['sem'],
                            binned['mean'] + binned['sem'],
                            alpha=0.2, color=color)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Tyre Life (laps on current set)')
    ax.set_title(f'{year} Season')
    ax.legend(title='Compound')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%+.2f'))

axes[0].set_ylabel('Lap Time Delta vs. Stint Median (s)')
fig.suptitle('Tyre Degradation Curves by Compound — 2024 vs. 2025', fontsize=13)
plt.tight_layout()
plt.savefig('plot_03_degradation_curves.png', bbox_inches='tight')
plt.show()
print('\nInterpretation: SOFT tyres show the steepest degradation slope in both seasons.'
      ' Comparing 2024 vs. 2025 side-by-side will help reveal whether regulation or'
      ' compound spec changes affected degradation rates — useful context for the model.')

### 4.4 SOFT Compound Degradation by Circuit

Checking whether circuit type drives meaningful variation in degradation rates.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

soft_df = df[df['Compound'] == 'SOFT'].copy()
races = soft_df['Race'].unique()
palette = sns.color_palette('tab10', n_colors=len(races))

for race, color in zip(races, palette):
    subset = soft_df[soft_df['Race'] == race]
    year = subset['Year'].iloc[0]
    binned = subset.groupby('TyreLife')['LapTimeDelta'].mean().reset_index()
    binned = binned[(binned['TyreLife'] >= 2) & (binned['TyreLife'] <= 25)]
    if len(binned) >= 5:
        ax.plot(binned['TyreLife'], binned['LapTimeDelta'],
                label=f'{race} ({year})', color=color, linewidth=1.8, alpha=0.85)

ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('Tyre Life (laps on current set)')
ax.set_ylabel('Lap Time Delta (s)')
ax.set_title('SOFT Tyre Degradation by Circuit (2024–2025)')
ax.legend(title='Race', fontsize=8, loc='lower left')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%+.2f'))

plt.tight_layout()
plt.savefig('plot_04_degradation_by_circuit.png', bbox_inches='tight')
plt.show()
print('\nInterpretation: Degradation rates vary meaningfully by circuit.'
      ' High-energy circuits show faster drop-off vs. lower-energy street circuits.'
      ' This validates including circuit as a feature in the model.')

### 4.5 Correlation Matrix

In [ ]:
corr_features = [
    'LapTimeDelta', 'TyreLife', 'TyreLifeRatio', 'CompoundEncoded',
    'StintLength', 'LapNumber', 'AirTemp', 'TrackTemp'
]

corr_matrix = df[corr_features].dropna().corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Key Numerical Features (2024–2025)')
plt.tight_layout()
plt.savefig('plot_05_correlation_matrix.png', bbox_inches='tight')
plt.show()
print('\nInterpretation: TyreLife and TyreLifeRatio show the strongest correlation with'
      ' LapTimeDelta — confirming these are the primary predictors.'
      ' TrackTemp shows a modest positive correlation.'
      ' CompoundEncoded shows expected negative correlation with delta at high tyre life.')

## 5. Dataset Summary

In [ ]:
print('=== Final Dataset Summary ===')
print(f'Total clean laps:    {len(df):,}')
print(f'Seasons:             2024, 2025')
print(f'Races included:      {df["Race"].nunique()} ({list(df["Race"].unique())})')
print(f'Unique drivers:      {df["Driver"].nunique()}')
print(f'Compounds:           {list(df["Compound"].value_counts().index)}')
print()
print('Laps per season:')
print(df.groupby('Year').size().to_string())
print()
print('Laps per compound:')
print(df['Compound'].value_counts().to_string())
print()
print('Key feature stats:')
print(df[['TyreLife', 'LapTimeSec', 'LapTimeDelta', 'TyreLifeRatio']].describe().round(3).to_string())

## 6. Challenges

- **Weather data gaps**: Not all sessions return complete weather data through FastF1. Missing `AirTemp` / `TrackTemp` values will be imputed from circuit historical averages or dropped for the baseline model.
- **Safety car edge cases**: `TrackStatus` is the most reliable flag but some partially neutralized laps still slip through. A secondary z-score filter per lap number may improve this.
- **Stint boundary noise**: The first 1–2 laps of each stint sometimes show anomalously fast times (fresh tyres + open track). We may exclude lap 1 per stint from the degradation target.
- **Scale-up**: Current run covers 8 races for development speed. Full 2024 + 2025 pull (~50,000 clean laps) will be run before final modeling.

## 7. Next Steps

**Weeks 5–6 — Modeling (Kevin)**
1. Baseline Linear Regression: predict `LapTimeDelta` from `TyreLife`, `CompoundEncoded`, `TrackTemp`, `StintLength`, and circuit one-hot encoding
2. Time-based cross-validation: train on 2024 season, validate on 2025 season
3. Random Forest and XGBoost: compare RMSE and feature importances against baseline
4. Target: RMSE < 0.3s per lap on held-out 2025 races

**Weeks 7–8 — Pit-Stop Optimizer (Michael)**
5. Build race-simulation module: given predicted degradation curves, identify compound sequence and stint lengths minimizing total race time
6. Back-test against actual 2025 race outcomes — target: strategy within 2 laps of actual optimal on ≥60% of test races